# Kalam on AMI: GPU benchmark

Runs Kalam's exact pipeline (`app.run_pipeline`) on the AMI Meeting Corpus with a GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

Protocol, so the result is publishable:
1. Compare configurations on the **dev** meetings only.
2. Pick the best configuration by dev score.
3. Run the **test** set once with that configuration and report it, alongside the earlier baselines.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

In [ ]:
!git clone -q https://github.com/faisalafieh/kalam
%cd kalam
!git log --oneline -1

### Install
Uses the same pinned versions as the server, with the CUDA build of torch. Takes about 3 minutes.

In [ ]:
!pip install -q torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip install -q fastapi==0.115.6 python-multipart==0.0.20 faster-whisper==1.1.1 pyannote.audio==3.3.2 "huggingface_hub<1.0" matplotlib -r eval/requirements.txt

In [ ]:
# faster-whisper (CTranslate2) needs to find the cuBLAS / cuDNN libraries that came with torch.
import os, glob, site
libs = sorted({os.path.dirname(p) for sp in site.getsitepackages()
               for p in glob.glob(f"{sp}/nvidia/*/lib/*.so*")})
os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])

import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

### Hugging Face token
Needs a **read** token, with the terms accepted on
[pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) and
[pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0).

In [ ]:
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")
os.environ.update(DEVICE="cuda", COMPUTE_TYPE="float16", LANGUAGE="en", BEAM_SIZE="5")

## 1. Compare configurations on the dev set

In [ ]:
!python eval/prepare_ami.py --split dev

In [ ]:
import subprocess

def run(tag, manifest, **settings):
    env = {**os.environ, **{k: str(v) for k, v in settings.items()}}
    print(f"\n=== {tag}: {settings}")
    subprocess.run(["python", "eval/evaluate.py", "--manifest", manifest, "--tag", tag,
                    "--hardware", "Colab " + torch.cuda.get_device_name(0)], env=env, check=True)

DEV = "eval/data/ami/manifest_dev_headset.jsonl"
CONFIGS = {
    "dev-small":            dict(WHISPER_MODEL="small"),
    "dev-large-v3-turbo":   dict(WHISPER_MODEL="large-v3-turbo"),
    "dev-large-v3":         dict(WHISPER_MODEL="large-v3"),
    # ablation: same best-guess model, old segment-level speaker assignment
    "dev-large-v3-segment": dict(WHISPER_MODEL="large-v3", WORD_SPEAKERS="false"),
}
for tag, settings in CONFIGS.items():
    run(tag, DEV, **settings)

In [ ]:
import json
dev = {t: json.load(open(f"eval/results/{t}.json"))["summary"] for t in CONFIGS}
for t, s in dev.items():
    print(f"{t:24s} WER {100*s['wer']:5.1f}%   cpWER {100*s['cpwer']:5.1f}%   DER {100*s['der']:5.1f}%   RTF {s['rtf']:.2f}")

BEST = min(dev, key=lambda t: dev[t]["cpwer"])   # selected on dev cpWER only
print("\nSelected on dev:", BEST, CONFIGS[BEST])

## 2. Final run on the full test set (once)

All 16 test meetings, about 9 hours of audio. Roughly an hour on a T4.

In [ ]:
!python eval/prepare_ami.py --all
run("test-final", "eval/data/ami/manifest_headset.jsonl", **CONFIGS[BEST])

## 3. Download the results
Commit `eval/RESULTS.md` and the `eval/results/*.json` files to the repo (these contain no audio).

In [ ]:
!zip -q -j kalam_results.zip eval/RESULTS.md eval/results/*.json
from google.colab import files
files.download("kalam_results.zip")